### Main paper plot 1

X-axis: greedy iterations

Y-axis: F(S, Q)

Each curve represents an algorithm: exact greedy, or augmented greedy with R sampled hyperplanes

In [1]:
import torch
import pickle
import os
import numpy as np

import plotly
import plotly.graph_objects as go

In [2]:
from plot_utils import crop_pdf_with_fitz, crop_pdf_with_pdfcrop, crop_pdf_with_pypdf

In [3]:
greedy_iters = 10
num_hyperplanes = 5

In [4]:
# scifact data
dataset = "scifact"
with open(f"./pickles/results/greedy_base_0_128_k15_{dataset}_bf.pkl", "rb") as f:
    scifact_greedy_data = pickle.load(f)

scifact_greedy_aug_data = []

for i in range(1, 9):
    with open(f"./pickles/results/greedy_base_0_128_k10_{dataset}_bf_augmented_{i}.pkl", "rb") as f:
        data = pickle.load(f)
    scifact_greedy_aug_data.append(data)

In [5]:
# nfcorpus data
dataset = "nfcorpus"
with open(f"./pickles/results/greedy_base_0_128_k15_{dataset}_bf.pkl", "rb") as f:
    nfcorpus_greedy_data = pickle.load(f)

nfcorpus_greedy_aug_data = []

for i in range(1, 9):
    with open(f"./pickles/results/greedy_base_0_128_k10_{dataset}_bf_augmented_{i}.pkl", "rb") as f:
        data = pickle.load(f)
    nfcorpus_greedy_aug_data.append(data)

In [29]:
# writing data
dataset = "writing"
with open(f"./pickles/results/greedy_base_0_128_k10_{dataset}_bf.pkl", "rb") as f:
    writing_greedy_data = pickle.load(f)

writing_greedy_aug_data = []

for i in range(1, 8):
    with open(f"./pickles/results/greedy_base_0_128_k10_{dataset}_bf_augmented_{i}.pkl", "rb") as f:
        data = pickle.load(f)
    writing_greedy_aug_data.append(data)

In [6]:
# scifact plotting generation
# Try query 0 for now

query_id = 0

num_curves = list(range(1, greedy_iters + 1))
y_axis_point_lists = []

for i in range(greedy_iters):
    greedy_score = np.array([scifact_greedy_data[query_id][i][1]])
    greedy_aug_scores = np.array([scifact_greedy_aug_data[j][query_id][i][1] for j in range(num_hyperplanes)])

    abs_diff = np.abs(greedy_aug_scores - greedy_score)

    print(abs_diff)
    y_axis_point_lists.append(abs_diff)

In [7]:
assert len(y_axis_point_lists) == greedy_iters

In [8]:
# Each curve on the plot corresponds to one greedy iteration.
# Each point on the curve corresponds to one hyperplane.
# So, for greedy iteration i, we have num_hyperplanes points.
# So, we need to plot greedy_iters curves, each with num_hyperplanes points.
# x-axis: # samples (number of hyperplanes)
# y-axis: absolute difference in scores
fig = go.Figure()
for i in range(greedy_iters):
    fig.add_trace(go.Scatter(x=list(range(1, num_hyperplanes + 1)), y=y_axis_point_lists[i], mode='lines+markers', name=f'Greedy Iteration {i+1}'))
fig.update_layout(title='SCIFACT: Absolute Difference in Scores vs Number of Hyperplanes',
                  xaxis_title='Number of Hyperplanes',
                  yaxis_title='Absolute Difference in Scores')
fig.update_layout(
    xaxis=dict(
        tickvals=list(range(1, num_hyperplanes + 1))  # Enforce integer ticks
    )
)
fig.show()

In [9]:
# nfcorpus plotting generation
# Try query 0 for now

query_id = 0

num_curves = list(range(1, greedy_iters + 1))
y_axis_point_lists = []

for i in range(greedy_iters):
    greedy_score = np.array([nfcorpus_greedy_data[query_id][i][1]])
    greedy_aug_scores = np.array([nfcorpus_greedy_aug_data[j][query_id][i][1] for j in range(num_hyperplanes)])

    abs_diff = np.abs(greedy_aug_scores - greedy_score)

    print(abs_diff)
    y_axis_point_lists.append(abs_diff)

In [10]:
assert len(y_axis_point_lists) == greedy_iters

In [11]:
# Each curve on the plot corresponds to one greedy iteration.
# Each point on the curve corresponds to one hyperplane.
# So, for greedy iteration i, we have num_hyperplanes points.
# So, we need to plot greedy_iters curves, each with num_hyperplanes points.
# x-axis: # samples (number of hyperplanes) (integers)
# y-axis: absolute difference in scores
# Ensure that the X-axis is integers from 1 to num_hyperplanes
fig = go.Figure()
for i in range(greedy_iters):
    fig.add_trace(go.Scatter(
        x=list(range(1, num_hyperplanes + 1)),
        y=y_axis_point_lists[i],
        mode='lines+markers', name=f'Greedy Iteration {i+1}'
    ))
fig.update_layout(title='NFCORPUS: Absolute Difference in Scores vs Number of Hyperplanes',
                  xaxis_title='Number of Hyperplanes',
                  yaxis_title='Absolute Difference in Scores')
fig.update_layout(
    xaxis=dict(
        tickvals=list(range(1, num_hyperplanes + 1))  # Enforce integer ticks
    )
)
fig.show()

In [12]:
[x[1] for x in scifact_greedy_data[0][:greedy_iters]]

In [13]:
# Now, for scifact, let's plot number of greedy iterations vs the score (not the difference of scores)
# The score is stored in arrays scifact_greedy_data (list of tuples) and scifact_greedy_aug_data (list of 8 lists of tuples).
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, greedy_iters + 1)),
    y=[item[1] for item in scifact_greedy_data[0][:greedy_iters]],
    mode='lines+markers',
    name=f'Greedy'
))

for i in range(num_hyperplanes):
    scifact_greedy_aug = scifact_greedy_aug_data[i]
    fig.add_trace(go.Scatter(x=list(range(1, greedy_iters + 1)), y=[item[1] for item in scifact_greedy_aug[0][:greedy_iters]], mode='lines+markers', name=f'Hyperplane {i + 1}'))
fig.update_layout(title='SCIFACT: F(S) vs K',
                  xaxis_title='K',
                  yaxis_title='F(S)')
fig.update_layout(
    xaxis=dict(
        tickvals=list(range(1, greedy_iters + 1))  # Enforce integer ticks
    )
)
fig.show()

In [14]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, greedy_iters + 1)),
    y=[item[1] for item in nfcorpus_greedy_data[0][:greedy_iters]],
    mode='lines+markers',
    name=f'Greedy'
))

for i in range(num_hyperplanes):
    nfcorpus_greedy_aug = nfcorpus_greedy_aug_data[i]
    fig.add_trace(go.Scatter(x=list(range(1, greedy_iters + 1)), y=[item[1] for item in nfcorpus_greedy_aug[0][:greedy_iters]], mode='lines+markers', name=f'Hyperplane {i + 1}'))
fig.update_layout(title='NFCORPUS: F(S) vs K',
                  xaxis_title='K',
                  yaxis_title='F(S)')
fig.update_layout(
    xaxis=dict(
        tickvals=list(range(1, greedy_iters + 1))  # Enforce integer ticks
    )
)
fig.show()

### MATPLOTLIB + LaTeX

In [22]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm

In [23]:
legend_labels = [
    r'\textbf{Exact Greedy}',
    r'\textbf{\#samples R=1}',
    r'\textbf{\#samples R=2}',
    r'\textbf{\#samples R=3}',
    r'\textbf{\#samples R=4}',
    r'\textbf{\#samples R=5}'
]

legend_color_map = {
    legend_labels[0]: "black",
    legend_labels[1]: "#FF8C00",  # Dark Orange
    legend_labels[2]: "#FF6347",  # Tomato
    legend_labels[3]: "#FF4500",  # Orange Red
    legend_labels[4]: "#DC143C",  # Crimson
    legend_labels[5]: "#8B0000"   # Dark Red
}

legend_marker_map = {
    legend_labels[0]: "o",
    legend_labels[1]: "v",
    legend_labels[2]: "v",
    legend_labels[3]: "v",
    legend_labels[4]: "v",
    legend_labels[5]: "v",
}

In [55]:
# Reset and apply working mathtext-compatible style
def plot_paper(greedy_data, greedy_aug_data, dataset_name, solid_index=1, y_label=True):
    plt.clf()
    fig, ax = plt.subplots(figsize=(8, 6))
    markersize = 10

    plt.rcParams.update({
        'text.usetex': True,
        'text.latex.preamble': r'\usepackage{amsmath}',
        'font.family': 'serif',
        # 'font.size': 16,
        # 'axes.labelsize': 36,    # Increased axis labels (K and F(S,Q))
        # 'xtick.labelsize': 36,   # Reduced X-axis tick labels (1,2,3...)
        # 'ytick.labelsize': 36,   # Reduced Y-axis tick labels (numbers on Y)
        'figure.dpi': 300,
        'lines.markersize': markersize
    })

    # Plot greedy data
    ax.plot(
        list(range(1, greedy_iters + 1)),
        [item[1] for item in greedy_data[0][:greedy_iters]],
        label=legend_labels[0],
        color=legend_color_map[legend_labels[0]],
        marker=legend_marker_map[legend_labels[0]],
        linewidth=2,
        # markersize=14,
        alpha=0.3
    )

    # Plot augmented data
    for i in range(0, solid_index):
        ax.plot(
            list(range(1, greedy_iters + 1)),
            [item[1] for item in greedy_aug_data[i][0][:greedy_iters]],
            label=legend_labels[i + 1],
            color=legend_color_map[legend_labels[i + 1]],
            marker=legend_marker_map[legend_labels[i + 1]],
            linewidth=2,
            # markersize=10,
        )

    for i in range(solid_index, num_hyperplanes):
        ax.plot(
            list(range(1, greedy_iters + 1)),
            [item[1] for item in greedy_aug_data[i][0][:greedy_iters]],
            label=legend_labels[i + 1],
            color=legend_color_map[legend_labels[i + 1]],
            marker=legend_marker_map[legend_labels[i + 1]],
            linewidth=2,
            # markersize=10,
            alpha=0.5
        )

    # ax.set_title(f'{dataset_name}: F(S) vs K', fontsize=20)
    ax.set_xlabel(r'$\textbf{K}\rightarrow$', fontsize=48)      # Increased axis label size
    if y_label:
        ax.set_ylabel(r'$\textbf{Avg}\quad\pmb{F(S_K, Q)}$', fontsize=48) # Increased axis label size
    ax.set_xticks([i for i in range(1, greedy_iters + 1) if i % 2 == 0])
    # Explicitly set tick label sizes
    ax.tick_params(axis='x', labelsize=50)  # Smaller X-axis tick labels
    ax.tick_params(axis='y', labelsize=50)  # Smaller Y-axis tick labels
    # ax.legend(fontsize=12)
    ax.grid(True)

    # Get current x-tick values and format them with LaTeX bold
    xticks = ax.get_xticks()
    xticklabels = [fr'$\mathbf{{{int(v)}}}$' for v in xticks]
    ax.set_xticklabels(xticklabels)

    # Get current y-tick values and format them with LaTeX bold
    yticks = ax.get_yticks()
    yticklabels = [fr'$\mathbf{{{int(v)}}}$' for v in yticks]
    ax.set_yticklabels(yticklabels)

    plt.tight_layout()
    plt.savefig(f'./notebooks/plots/{dataset_name.lower()}_plot1.pdf')
    plt.show()

In [32]:
def plot_legend_only(greedy_data, greedy_aug_data, filename, ncol=3, legend_order=None, auto_crop=True):
    num_hyperplanes = len(greedy_data) - 1  # Exclude the greedy data itself
    
    if legend_order is None:
        legend_order = list(range(num_hyperplanes))

    # Create a figure for legend only
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plot greedy data
    ax.plot(
        [],
        [],
        label=legend_labels[0],
        color=legend_color_map[legend_labels[0]],
        marker=legend_marker_map[legend_labels[0]],
        linewidth=2,
        markersize=10,
        alpha=0.3
    )

    for i in legend_order:
        ax.plot(
            [],
            [],
            label=legend_labels[i + 1],
            color=legend_color_map[legend_labels[i + 1]],
            marker=legend_marker_map[legend_labels[i + 1]],
            linewidth=2,
            markersize=10,
        )

    # Hide the plot area
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Create legend
    legend = ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=ncol,
        fontsize=14,
        frameon=False
    )

    # Save the figure
    output_path = f'./notebooks/plots/{filename}.pdf'
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()
    
    # Auto-crop the PDF if requested
    if auto_crop:
        # Try methods in order of preference
        cropped_path = crop_pdf_with_pdfcrop(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_fitz(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_pypdf(output_path)
        
        if cropped_path:
            # Replace original with cropped version
            os.rename(cropped_path, output_path)
            print(f"PDF automatically cropped: {output_path}")
        else:
            print("Auto-cropping failed, using matplotlib's bbox_inches='tight' only")

In [33]:
plot_legend_only(scifact_greedy_data, scifact_greedy_aug_data, "plot1_legend", legend_order=[2, 0, 3, 1, 4])

In [56]:
plot_paper(scifact_greedy_data, scifact_greedy_aug_data, "scifact", solid_index=3, y_label=False)

In [57]:
plot_paper(nfcorpus_greedy_data, nfcorpus_greedy_aug_data, "nfcorpus", solid_index=2)

In [58]:
plot_paper(writing_greedy_data, writing_greedy_aug_data, "writing", solid_index=4, y_label=False)